# Gaussian Splatting with nerfstudio

This notebook walks you through the full gaussian splatting pipeline:

1. **Upload** a video you've taken
2. **Process** the video (extract frames, estimate camera positions)
3. **Train** a gaussian splat model
4. **View** the result in an interactive viewer
5. **Export** a .ply file you can use in SuperSplat, Three.js, Blender, etc.

**Before you start:** Make sure the kernel is set to **nerfstudio** (top right of the notebook, or Kernel → Change Kernel).

## Step 0: Setup

Run this cell once to verify nerfstudio is installed and the GPU is available.

In [ ]:
import subprocess
import os
import sys
from pathlib import Path
from IPython.display import display, Markdown

# Ensure the conda environment's bin/ is on PATH so shell commands
# (ns-train, ns-process-data, ns-export, colmap) are found.
env_bin = os.path.join(sys.prefix, "bin")
if env_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = env_bin + os.pathsep + os.environ.get("PATH", "")

# Check nerfstudio is available
result = subprocess.run(["ns-train", "--help"], capture_output=True, text=True)
if result.returncode == 0:
    print("nerfstudio is installed.")
else:
    print("ERROR: nerfstudio not found. Make sure you're using the 'nerfstudio' kernel.")

# Check GPU
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode == 0:
    print(f"GPU: {gpu_check.stdout.strip()}")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

## Step 1: Set your video path

Upload your video file to this JupyterHub workspace (drag it into the file browser on the left), then set the path below.

**Tips for good captures:**
- Walk slowly around the subject in a smooth arc or circle
- Keep the subject centered in frame
- Aim for 30-60 seconds for an object, 1-2 minutes for a room
- Avoid fast movements, motion blur, and drastic lighting changes

In [ ]:
#
# CHANGE THIS to the filename of your uploaded video
#
VIDEO_FILENAME = "my-video.mp4"

# Project name (used for output folder names — no spaces)
PROJECT_NAME = "my-splat"

# ----- You shouldn't need to change anything below this line -----

VIDEO_PATH = Path(VIDEO_FILENAME).resolve()
PROCESSED_DIR = Path(f"data/{PROJECT_NAME}/processed")
OUTPUT_DIR = Path(f"data/{PROJECT_NAME}/output")
EXPORT_DIR = Path(f"data/{PROJECT_NAME}/export")

if not VIDEO_PATH.exists():
    print(f"ERROR: Video not found at {VIDEO_PATH}")
    print(f"Make sure '{VIDEO_FILENAME}' is uploaded to your JupyterHub workspace.")
else:
    size_mb = VIDEO_PATH.stat().st_size / (1024 * 1024)
    print(f"Video: {VIDEO_PATH} ({size_mb:.1f} MB)")
    print(f"Project: {PROJECT_NAME}")
    print(f"Output will be saved to: data/{PROJECT_NAME}/")

## Step 2: Process video

This step extracts frames from your video and estimates where the camera was positioned for each frame. It uses **hloc** (SuperPoint + SuperGlue) for feature matching, which is more robust than COLMAP's default SIFT pipeline, then COLMAP for the final reconstruction.

**This may take a few minutes** depending on video length.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

!ns-process-data video \
    --data "{VIDEO_PATH}" \
    --output-dir "{PROCESSED_DIR}" \
    --sfm-tool hloc

# Check results
images_dir = PROCESSED_DIR / "images"
if images_dir.exists():
    num_frames = len(list(images_dir.glob("*.png"))) + len(list(images_dir.glob("*.jpg")))
    print(f"\nProcessing complete! Extracted {num_frames} frames.")
else:
    print("\nERROR: Processing may have failed. Check the output above for errors.")

## Step 3: Train the gaussian splat

This trains a splatfacto model — nerfstudio's gaussian splatting implementation. The model learns to represent your scene as millions of tiny colored 3D gaussians.

**Training time:** ~5-10 minutes at the default 7,000 iterations. You can increase `MAX_ITERATIONS` below for higher quality (at the cost of longer training time — 30,000 iterations takes ~20-30 minutes).

During training, a Viser viewer URL will be printed. You can open it in a new browser tab to watch the splat take shape in real time.

In [ ]:
import random
MAX_ITERATIONS = 7000  # Increase for higher quality (e.g. 15000 or 30000)
VIEWER_PORT = random.randint(10000, 65000)

print(f"Opening Viser viewer at http://itp-ml.itp.tsoa.nyu.edu:{VIEWER_PORT}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

!ns-train splatfacto \
    --data "{PROCESSED_DIR}" \
    --output-dir "{OUTPUT_DIR}" \
    --max-num-iterations {MAX_ITERATIONS} \
    --viewer.quit-on-train-completion False \
    --viewer.websocket-port {VIEWER_PORT}

## Step 4: Find the config file

nerfstudio saves training output in a timestamped folder. This cell finds the most recent config file automatically.

In [ ]:
config_files = sorted(OUTPUT_DIR.rglob("config.yml"), key=os.path.getmtime, reverse=True)

if not config_files:
    print("ERROR: No config.yml found. Training may not have completed.")
    CONFIG_PATH = None
else:
    CONFIG_PATH = config_files[0]
    print(f"Using config: {CONFIG_PATH}")

## Step 5: Export as .ply

This exports your trained gaussian splat as a .ply file. You can then:
- Download it and open it in [SuperSplat](https://playcanvas.com/super-splat) for editing
- Use it in the Three.js splat viewer template
- Import it into Blender or Unity

In [ ]:
if CONFIG_PATH:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    !ns-export gaussian-splat \
        --load-config "{CONFIG_PATH}" \
        --output-dir "{EXPORT_DIR}"

    ply_files = list(EXPORT_DIR.glob("*.ply"))
    if ply_files:
        ply_path = ply_files[0]
        size_mb = ply_path.stat().st_size / (1024 * 1024)
        print(f"\nExported: {ply_path} ({size_mb:.1f} MB)")
        print(f"\nRight-click the file in the JupyterHub file browser and select 'Download' to save it to your computer.")
    else:
        print("ERROR: No .ply file found in export directory.")
else:
    print("Skipping export — no config file found. Run Step 4 first.")

## Step 6 (optional): Launch the viewer

If you want to explore the trained splat interactively without exporting first, run the cell below to launch the Viser viewer. Open the printed URL in a new browser tab.

In [ ]:
import random
VIEWER_PORT = random.randint(10000, 65000)

print(f"Opening Viser viewer at http://itp-ml.itp.tsoa.nyu.edu:{VIEWER_PORT}")

if CONFIG_PATH:
    !ns-viewer --load-config "{CONFIG_PATH}" --viewer.websocket-port {VIEWER_PORT}
else:
    print("No config file found. Run Steps 3 and 4 first.")